# vLLM
原生 `transformers` 库中，模型默认采用 `sequential` 顺序推导（即单请求串行处理），并且由于缺乏对 KV Cache 显存的高效管理，在大并发场景下极易发生显存碎片化（OOM）与吞吐量瓶颈。
vLLM生产级 LLM 推理加速的行业标准引擎。

## vLLM 加速原理与高并发接口服务部署
1. PagedAttention 核心原理：掌握 vLLM 如何借鉴操作系统虚拟内存页（Paging）的思想，消除 KV Cache 的显存碎片化。
2. 掌握 Continuous Batching（连续批处理）：理解动态迭代级批处理机制如何提升 GPU 利用率与吞吐量（Throughput）。
3. 使用 vLLM 部署 OpenAI 兼容的 API Server 并使用 Python 进行高并发流式调用。

#### 经典 Transformer 推理的显存痛点：KV Cache 碎片化
在 LLM 的自回归生成过程中，为了避免重复计算历史 Token 的 Key 和 Value 向量，系统会在显存中维护 KV Cache。但在传统框架（如原生 HF `transformers`）中：
* 显存预分配浪费：由于无法预知模型最终会输出多少个 Token，系统通常必须按照 `max_sequence_length` 预先分配连续的物理显存，导致大量未使用的显存被空置占用。
* 显存碎片化（Memory Fragmentation）：不同请求的序列长度不同，频繁地申请与释放连续物理显存会导致严重的内存碎片，极大地限制了最大并发数（Batch Size）。

#### PagedAttention分页的破局方案
vLLM 借鉴了现代操作系统（OS）的分页虚拟内存（Paged Virtual Memory）思想：
* KV 块（KV Blocks）映射：将连续的逻辑 Key/Value 向量切分为固定大小的 Block（如每块包含 16 个 Token 的 KV 向量）。
* Block Table 物理映射：将逻辑上连续的 Token Block 映射到物理显存中不连续的物理内存块上。
* 零浪费动态按需分配：只有当新 Token 生成并填满当前 Block 时，才向显存池申请一个新的 Block，将 KV Cache 的显存浪费率直接降低到 1% 以下！

#### Continuous Batching（连续批处理 / 迭代级批处理）
传统 Batching 需要等待当前 Batch 中所有序列都生成完毕后才能处理下一批请求（被短序列拖累）。
vLLM 的 Continuous Batching 可以在每一个 Generation Step（Token 级迭代）完成后，动态地将已完成的请求移出 Batch，并将等待队列中的新请求插入 Batch，使 GPU 计算单元始终处于饱满的高吞吐状态。

## vLLM 部署 API 并进行高并发流式调用
环境准备提示：运行本段代码需要安装 vLLM 官方库（建议在 Linux 环境或带有 NVIDIA GPU 的 Docker 容器中运行）：
`pip install vllm openai`

#### 步骤 1：命令行启动 vLLM OpenAI 兼容 Server（服务端）
在终端中执行以下命令，vLLM 会自动加载模型并暴露与 OpenAI 规范完全一致的 RESTful API 端口：

In [ ]:
python3 -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct \
    --dtype bfloat16 \
    --gpu-memory-utilization 0.85 \
    --max-model-len 4096 \
    --port 8000

#### 步骤 2：使用 OpenAI Python SDK 进行高并发流式调用（客户端）
由于 vLLM 完全兼容 OpenAI 的接口协议，我们可以直接使用官方 `openai` SDK 连接本地 vLLM 服务，无需改变现有 Agent 系统中的任何调用代码！

In [ ]:
import async_timeout
import asyncio
from openai import AsyncOpenAI

# 指向本地 vLLM 启动的 API 端口
client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY"  # vLLM 本地服务默认不需要真实 API Key
)

async def generate_response(prompt_id: int, user_prompt: str):
    """模拟并发请求 vLLM 推理服务"""
    print(f"🚀 请求 #{prompt_id} 已发送...")

    response = await client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[
            {"role": "system", "content": "你是一名资深的 Python 高性能计算专家。"},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=256,
        stream=True  # 启用流式响应
    )

    full_text = ""
    async for chunk in response:
        if chunk.choices and chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            full_text += content

    print(f"✅ 请求 #{prompt_id} 接收完成！首句内容: {full_text[:30].strip()}...")

async def main():
    prompts = [
        "请解释什么是 vLLM 的 PagedAttention 机制？",
        "用 Python 实现一个高并发的协程池处理逻辑。",
        "对比 Redis 与 Memcached 的内存管理策略异同。"
    ]

    # 使用 asyncio.gather 并发发起 3 个请求，验证 Continuous Batching 特性
    print("🔥 开始高并发测试（Continuous Batching）...\n")
    await asyncio.gather(
        *(generate_response(i + 1, prompt) for i, prompt in enumerate(prompts))
    )

if __name__ == "__main__":
    asyncio.run(main())

**核心工程细节解析**
1. `--gpu-memory-utilization 0.85`：控制 vLLM 预先预留给 KV Cache 池的显存比例（默认 0.90）。设置 0.85 意味着保留 15% 的 GPU 显存留给其他模型临时加载或 PyTorch 算子开销，防止突发 OOM。
2. 零成本迁移现有的 OpenAI / LangChain 代码：通过配置 `base_url="http://localhost:8000/v1"`，你之前写的 LangChain、LangGraph 或 AutoGen 项目无需改动底层逻辑，只需修改 API 域名即可直接无缝无缝对接本地 vLLM 高并发推理集群。
3. 高吞吐（Throughput）与低延迟（Latency）的权衡：vLLM 的优势在于大幅提升系统的每秒吞吐 Token 数（Throughput）。对于单个请求的首字延迟（Time to First Token, TTFT），PagedAttention 会维持极其优秀的毫秒级响应。

**思考**
1. 多卡张量并行（Tensor Parallelism）：如果你手头有 2 张 RTX 4090 GPU，想要将一个 70B 的大模型切片部署在两张卡上，应该在启动 vLLM 命令行时加入哪个参数（例如 `--tensor-parallel-size`）？它的底层通信使用的是 PyTorch 的哪种原语？
2. 前向推理优化：在部署大模型服务时，开启 vLLM 的 `--enable-chunked-prefill` 参数对处理长 Prompt（超长 Context Window）时的 TTFT（首字延迟）有何优化效果？